In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

Pkg.instantiate()

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

# Една зависима променлива => една JetSpace променлива
localsp = TS.JetSpace(order=20, variables=[:x])

println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))
println("IntervalArithmetic: ", Base.pkgversion(IntervalArithmetic))
println("JetSpace variables: ", TS.get_numvars(localsp))

@assert TS.get_numvars(localsp) == 1

function test_rhs!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

x0 = [interval(1.0)]

println("\\nStarting validated ODE test...")

sol = VI.validated_integ3(
    test_rhs!,
    x0,
    0.0,
    1.0,
    20,
    20,
    1e-12,
    localsp
)

println("\\nINTEGRATION COMPLETED")
println("Result type: ", typeof(sol))
println("Fields: ", fieldnames(typeof(sol)))
println("Number of time points: ", length(sol.time))
println("Final time: ", sol.time[end])

Y = sol.fp[end][1]
E = exp(interval(1.0))

println("\\nValidated final enclosure:")
println(Y)

println("\\nIndependent exp(1) interval:")
println(E)

println("\\nDoes the final enclosure contain exp(1) interval?")
println(inf(Y) <= inf(E) && sup(E) <= sup(Y))

println("\\nSMOKE TEST FINISHED")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


Julia: 1.11.9
TaylorModels: 0.13.1
IntervalArithmetic: 1.0.12
JetSpace variables: 1

Starting validated ODE test...

INTEGRATION COMPLETED
Result type: TaylorModels.ValidatedInteg.TMSol3{1, Float64, Float64}
Fields: (:time, :fp, :xTM)
Number of time points: 3
Final time: 0.5

Validated final enclosure:
[-6.3957e-20, 3.85265e-18]_com_NG

Independent exp(1) interval:
[2.71828, 2.71829]_com

Does the final enclosure contain exp(1) interval?
false

SMOKE TEST FINISHED


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg

localsp = TaylorModels.TaylorSeries.JetSpace(
    order=20,
    variables=[:x]
)

function test_rhs!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

x0 = [interval(1.0)]

sol = VI.validated_integ3(
    test_rhs!,
    x0,
    0.0,
    1.0,
    20,
    20,
    1e-12,
    localsp
)

println("EXPANSION TIMES:")
println(sol.time)

println("\\nNUMBER OF FLOWPIPE STEPS:")
println(length(sol.fp))

println("\\nTAYLOR MODEL MATRIX SIZE:")
println(size(sol.xTM))

for k in axes(sol.xTM, 2)
    tm = sol.xTM[1,k]

    println("\\nSTEP: ", k)
    println("Expansion point: ", sol.time[k])
    println("Taylor model domain: ", tm.dom)

    B = TaylorModels.bound_taylor1(tm)
    println("Validated range: ", B)
end

Y = TaylorModels.bound_taylor1(sol.xTM[1,end])
E = exp(interval(1.0))

println("\\nFINAL TAYLOR MODEL RANGE:")
println(Y)

println("\\nREFERENCE exp(1):")
println(E)

println("\\nINCLUSION CHECK:")
println(inf(Y) <= inf(E) && sup(E) <= sup(Y))

println("\\nEND OF DIAGNOSTIC")
""")

EXPANSION TIMES:
[0.0, 0.0, 0.5]

NUMBER OF FLOWPIPE STEPS:
3

TAYLOR MODEL MATRIX SIZE:
(1, 3)

STEP: 1
Expansion point: 0.0
Taylor model domain: [0.0, 0.0]_com


  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


JuliaError: MethodError: no method matching differentiate!(::Taylor1{TaylorModelN{1, Float64, Float64}}, ::Taylor1{TaylorModelN{1, Float64, Float64}}, ::Int64)
The function `differentiate!` exists, but no method is defined for this combination of argument types.

Closest candidates are:
  differentiate!(::Taylor1, ::Taylor1)
   @ TaylorSeries ~/.julia/packages/TaylorSeries/WXyOV/src/calculus.jl:40
  differentiate!(!Matched::Taylor1{Taylor1{T}}, !Matched::Taylor1{Taylor1{T}}, ::Int64) where T<:Union{Real, Complex}
   @ TaylorSeries ~/.julia/packages/TaylorSeries/WXyOV/src/calculus.jl:67
  differentiate!(!Matched::Taylor1{Taylor1{T}}, !Matched::Taylor1{Taylor1{T}}, ::Int64) where T<:Union{Real, Complex, Taylor1}
   @ TaylorSeries ~/.julia/packages/TaylorSeries/WXyOV/src/calculus.jl:84
  ...

Stacktrace:
 [1] differentiate
   @ ~/.julia/packages/TaylorSeries/WXyOV/src/calculus.jl:21 [inlined]
 [2] bound_taylor1(fT::Taylor1{TaylorModelN{1, Float64, Float64}}, I::Interval{Float64})
   @ Tay

In [ ]:

%pip install -q juliacall

import juliacall
from juliacall import Main as jl

print("juliacall version:", juliacall.__version__)

jl.seval("""
println("Julia connection: SUCCESS")
println("Julia version: ", VERSION)
println("Julia executable: ", Sys.BINDIR)
""")

[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliacall/juliapkg.json
[juliapkg] Found dependencies: /usr/local/lib/python3.13/dist-packages/juliapkg/juliapkg.json
[juliapkg] Locating Julia 1.10.3 - 1.11
[juliapkg] WARNING: You have Julia 1.12.6 installed but 1.10.3 - 1.11 is required.
[juliapkg]   It is recommended that you upgrade Julia or install JuliaUp.
[juliapkg] Querying Julia versions from https://julialang-s3.julialang.org/bin/versions.json
[juliapkg] WARNING: About to install Julia 1.11.9 to /root/.julia/environments/pyjuliapkg/pyjuliapkg/install.
[juliapkg]   If you use juliapkg in more than one environment, you are likely to
[juliapkg]   have Julia installed in multiple locations. It is recommended to
[juliapkg]   install JuliaUp (https://github.com/JuliaLang/juliaup) or Julia
[juliapkg]   (https://julialang.org/downloads) yourself.
[juliapkg] Downloading Julia from https://julialang-s3.julialang.org/bin/linux/x64/1.11/julia-1.11.9-linux-x86_64.tar.

In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

localsp = TS.JetSpace(order=20, variables=[:x])

function test_rhs!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sol = VI.validated_integ3(
    test_rhs!,
    [interval(1.0)],
    0.0,
    1.0,
    20,
    20,
    1e-12,
    localsp
)

println("EXPANSION TIMES:")
println(sol.time)

println("\\nVALIDATED TIME DOMAINS:")
domains = VI.mince_in_time(sol; var=0, timediv=1)
println(domains)

E = exp(interval(1.0))

println("\\nANALYTICAL exp(1) ENCLOSURE:")
println(E)

println("\\nFLOWPIPE INTERVALS:")

for k in eachindex(sol.fp)
    Y = sol.fp[k][1]

    println("\\nSTEP: ", k)
    println("Interval: ", Y)
    println("Contains exp(1): ",
        inf(Y) <= inf(E) && sup(E) <= sup(Y))
end

println("\\nNUMBER OF FLOWPIPE STEPS: ", length(sol.fp))
println("DIAGNOSTIC FINISHED")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`
┌ Warning: Assignment to `Y` in soft scope is ambiguous because a global variable by the same name exists: `Y` will be treated as a new local. Disambiguate by using `local Y` to suppress this warning or `global Y` to assign to the existing global variable.
└ @ none:45


EXPANSION TIMES:
[0.0, 0.0, 0.5]

VALIDATED TIME DOMAINS:
Interval{Float64}[[0.0, 0.0]_com_NG, [0.0, 0.5]_com_NG, [0.5, 1.0]_com_NG]

ANALYTICAL exp(1) ENCLOSURE:
[2.71828, 2.71829]_com

FLOWPIPE INTERVALS:

STEP: 1
Interval: [1.0, 1.0]_com
Contains exp(1): false

STEP: 2
Interval: [0.999999, 1.64873]_com_NG
Contains exp(1): false

STEP: 3
Interval: [-6.3957e-20, 3.85265e-18]_com_NG
Contains exp(1): false

NUMBER OF FLOWPIPE STEPS: 3
DIAGNOSTIC FINISHED


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

localsp = TS.JetSpace(order=20, variables=[:x])

function test_rhs!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sol = VI.validated_integ3(
    test_rhs!,
    [interval(1.0)],
    0.0,
    1.0,
    20,
    20,
    1e-12,
    localsp
)

println("TIME DOMAINS:")
println(VI.mince_in_time(sol; var=0, timediv=1))

println("\\nEVALUATED FLOWPIPE FOR VARIABLE 1:")
R = VI.mince_in_time(sol; var=1, timediv=1)

println("Result type: ", typeof(R))
println("Result:")
dump(R, maxdepth=4)

println("\\nTAYLOR MODEL STRUCTURE:")
println("xTM type: ", typeof(sol.xTM))
println("xTM size: ", size(sol.xTM))

for k in axes(sol.xTM, 2)
    println("\\nTaylor model step ", k)
    tm = sol.xTM[1,k]
    println("Type: ", typeof(tm))
    println("Fields: ", fieldnames(typeof(tm)))
    dump(tm, maxdepth=3)
end

println("\\nREFERENCE VALUES:")
println("exp(0.0) = ", exp(0.0))
println("exp(0.5) = ", exp(0.5))
println("exp(1.0) = ", exp(1.0))

println("\\nDIAGNOSTIC COMPLETE")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


TIME DOMAINS:
Interval{Float64}[[0.0, 0.0]_com_NG, [0.0, 0.5]_com_NG, [0.5, 1.0]_com_NG]

EVALUATED FLOWPIPE FOR VARIABLE 1:
Result type: Vector{Interval{Float64}}
Result:
Array{Interval{Float64}}((3,)) Interval{Float64}[[1.0, 1.0]_com, [0.999999, 1.64873]_com_NG, [-6.3957e-20, 3.85265e-18]_com_NG]

TAYLOR MODEL STRUCTURE:
xTM type: Matrix{TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}}
xTM size: (1, 3)

Taylor model step 1
Type: TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}
Fields: (:pol, :rem, :x0, :dom)
TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}
  pol: Taylor1{TaylorModelN{1, Float64, Float64}}
    coeffs: FixedSizeArrays.FixedSizeArrayDims{TaylorModelN{1, Float64, Float64}, 1, Tuple{Int64}, Memory{TaylorModelN{1, Float64, Float64}}}
      mem: Memory{TaylorModelN{1, Float64, Float64}}
      dims: Tuple{Int64}
  rem: Interval{Float64}
    bareinterval: BareInterval{Float64}
      lo: Float64 0.0
      hi: Float64 0.0
    decoration: Decoration com


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

println("TAYLOR MODEL EVALUATION TEST")
println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))

for k in axes(sol.xTM, 2)
    tm = sol.xTM[1,k]

    # Стъпка 1 е началната точка.
    # Стъпки 2 и 3 имат локална времева област [0, 0.5].
    tau = k == 1 ? 0.0 : 0.5

    println("\\n------------------------------")
    println("STEP: ", k)
    println("Expansion time: ", sol.time[k])
    println("Taylor domain: ", tm.dom)
    println("Evaluation tau: ", tau)
    println("Taylor remainder: ", tm.rem)
    println("Remainder guaranteed: ", isguaranteed(tm.rem))

    try
        # Оценка на полинома по локалното време.
        P = tm.pol(tau)

        println("Polynomial evaluation type: ", typeof(P))

        # Интервална оценка по началните параметри.
        B = TaylorModels.monotonicity_bounder(P)

        println("Polynomial range: ", B)
        println("Polynomial range guaranteed: ", isguaranteed(B))

        # Добавяме външния остатък на TaylorModel1.
        R = B + tm.rem

        println("TOTAL ENCLOSURE: ", R)
        println("Total enclosure guaranteed: ", isguaranteed(R))

        E = exp(interval(k == 3 ? 1.0 : 0.5))

        println("Reference exponential: ", E)
        println("Contains reference: ",
            inf(R) <= inf(E) && sup(E) <= sup(R))

    catch err
        println("EVALUATION ERROR: ", typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("\\nEND OF TAYLOR MODEL EVALUATION TEST")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`
┌ Warning: Assignment to `R` in soft scope is ambiguous because a global variable by the same name exists: `R` will be treated as a new local. Disambiguate by using `local R` to suppress this warning or `global R` to assign to the existing global variable.
└ @ none:40
┌ Warning: Assignment to `E` in soft scope is ambiguous because a global variable by the same name exists: `E` will be treated as a new local. Disambiguate by using `local E` to suppress this warning or `global E` to assign to the existing global variable.
└ @ none:45


TAYLOR MODEL EVALUATION TEST
Julia: 1.11.9
TaylorModels: 0.13.1

------------------------------
STEP: 1
Expansion time: 0.0
Taylor domain: [0.0, 0.0]_com
Evaluation tau: 0.0
Taylor remainder: [0.0, 0.0]_com
Remainder guaranteed: true
Polynomial evaluation type: TaylorModelN{1, Float64, Float64}
Polynomial range: [1.0, 1.0]_trv_NG
Polynomial range guaranteed: false
TOTAL ENCLOSURE: [1.0, 1.0]_trv_NG
Total enclosure guaranteed: false
Reference exponential: [1.64872, 1.64873]_com
Contains reference: false

------------------------------
STEP: 2
Expansion time: 0.0
Taylor domain: [0.0, 0.5]_com_NG
Evaluation tau: 0.5
Taylor remainder: [-1.16286e-19, 7.00481e-18]_com_NG
Remainder guaranteed: false
Polynomial evaluation type: TaylorModelN{1, Float64, Float64}
Polynomial range: [1.64872, 1.64873]_trv_NG
Polynomial range guaranteed: false
TOTAL ENCLOSURE: [1.64872, 1.64873]_trv_NG
Total enclosure guaranteed: false
Reference exponential: [1.64872, 1.64873]_com
Contains reference: true

--------

In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg

println("RECONSTRUCT RHO TEST")
println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))

println("\\nSTEP DOMAINS:")
for k in 1:length(sol.fp)
    println("Step ", k, ": ", VI.domain(sol, k))
end

println("\\nRECONSTRUCTING RHO...")
try
    rho = VI.reconstruct_rho(sol)

    println("RECONSTRUCT_RHO: SUCCESS")
    println("Type: ", typeof(rho))
    println("Size: ", size(rho))

    for k in axes(rho, 2)
        println("\\nSTEP ", k)
        for i in axes(rho, 1)
            R = TaylorModels.monotonicity_bounder(rho[i,k])
            println("rho[", i, ",", k, "] range: ", R)
            println("Guaranteed: ", isguaranteed(R))
            println("Inside [-1,1]: ",
                inf(R) >= -1 && sup(R) <= 1)
        end
    end

catch err
    println("RECONSTRUCT_RHO: FAILED")
    println("Error type: ", typeof(err))
    showerror(stdout, err)
    println()
end

println("\\nEND OF TEST")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


RECONSTRUCT RHO TEST
Julia: 1.11.9
TaylorModels: 0.13.1

STEP DOMAINS:
Step 1: [0.0, 0.0]_com
Step 2: [0.0, 0.5]_com_NG
Step 3: [0.0, 0.5]_com_NG

RECONSTRUCTING RHO...


┌ Warning: Assignment to `R` in soft scope is ambiguous because a global variable by the same name exists: `R` will be treated as a new local. Disambiguate by using `local R` to suppress this warning or `global R` to assign to the existing global variable.
└ @ none:30


RECONSTRUCT_RHO: FAILED
Error type: AssertionError
AssertionError: reconstruct_rho: L_{k+1} ≠ τ^0 coefficient of sol[3]
(non-deterministic, or different evaluation/preconditioning calls)

END OF TEST


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_reconstruct_test!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

println("FRESH ISOLATED TEST")
println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))
println("IntervalArithmetic: ", Base.pkgversion(IntervalArithmetic))

let
    sp = TS.JetSpace(order=20, variables=[:x])

    fresh_sol = VI.validated_integ3(
        rhs_reconstruct_test!,
        [interval(1.0)],
        0.0,
        1.0,
        20,
        20,
        1e-12,
        sp
    )

    println("\\nINTEGRATION FINISHED")
    println("Steps: ", length(fresh_sol.fp))
    println("Expansion times: ", fresh_sol.time)

    println("\\nIMMEDIATE RECONSTRUCTION")
    try
        rho = VI.reconstruct_rho(fresh_sol)

        println("RECONSTRUCT_RHO: SUCCESS")
        println("Size: ", size(rho))

        for k in axes(rho, 2)
            for i in axes(rho, 1)
                R = TaylorModels.monotonicity_bounder(rho[i,k])
                println("rho[", i, ",", k, "] = ", R)
                println("Guaranteed: ", isguaranteed(R))
            end
        end
    catch err
        println("RECONSTRUCT_RHO: FAILED")
        println("Error: ", typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("\\nISOLATED TEST COMPLETE")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


FRESH ISOLATED TEST
Julia: 1.11.9
TaylorModels: 0.13.1
IntervalArithmetic: 1.0.12

INTEGRATION FINISHED
Steps: 3
Expansion times: [0.0, 0.0, 0.5]

IMMEDIATE RECONSTRUCTION
RECONSTRUCT_RHO: FAILED
Error: AssertionError
AssertionError: reconstruct_rho: L_{k+1} ≠ τ^0 coefficient of sol[3]
(non-deterministic, or different evaluation/preconditioning calls)

ISOLATED TEST COMPLETE


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_diagnostic!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

sol_diag = VI.validated_integ3(
    rhs_diagnostic!,
    [interval(1.0)],
    0.0,
    1.0,
    20,
    20,
    1e-12,
    sp
)

println("=== SOLUTION STRUCTURE ===")
println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))
println("Time points: ", sol_diag.time)
println("Number of steps: ", length(sol_diag.fp))
println("xTM size: ", size(sol_diag.xTM))

println("\\n=== EACH TAYLOR MODEL ===")

for k in axes(sol_diag.xTM, 2)
    tm = sol_diag.xTM[1,k]

    println("\\nSTEP ", k)
    println("Expansion time: ", sol_diag.time[k])
    println("x0: ", tm.x0)
    println("dom: ", tm.dom)
    println("remainder: ", tm.rem)
    println("remainder guaranteed: ", isguaranteed(tm.rem))
    println("polynomial type: ", typeof(tm.pol))
    println("polynomial length: ", length(tm.pol))

    for j in 0:min(3, length(tm.pol)-1)
        coeff = tm.pol[j]
        println("coefficient[", j, "]: ", coeff)
        println("coefficient type: ", typeof(coeff))

        try
            println("coefficient range: ",
                TaylorModels.monotonicity_bounder(coeff))
        catch err
            println("coefficient range error: ", typeof(err))
        end
    end
end

println("\\n=== FLOWPIPE DATA ===")

for k in eachindex(sol_diag.fp)
    println("fp[", k, "] = ", sol_diag.fp[k])
end

println("\\n=== DOMAIN API ===")
for k in 1:length(sol_diag)
    println("domain(", k, ") = ", VI.domain(sol_diag, k))
end

println("\\n=== RECONSTRUCTION TEST ===")
try
    rho_diag = VI.reconstruct_rho(sol_diag)
    println("SUCCESS: ", size(rho_diag))
catch err
    println("FAILED: ", typeof(err))
    showerror(stdout, err)
    println()
end

println("\\nDIAGNOSTIC COMPLETE")
""")
Върни ми целия изход, включително грешките, ако има такива.

SyntaxError: invalid syntax (692376484.py, line 91)

In [ ]:


from juliacall import Main as jl

print("Python: OK")

jl.seval("""
println("Julia: OK")
println("Julia version: ", VERSION)
println("TaylorModels loaded: ", Base.find_package("TaylorModels") !== nothing)
""")

Python: OK
Julia: OK
Julia version: 1.11.9
TaylorModels loaded: true


In [ ]:



from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

println("=== PACKAGE DIAGNOSTICS ===")
println("Julia version: ", VERSION)
println("TaylorModels version: ",
    Base.pkgversion(TaylorModels))
println("IntervalArithmetic version: ",
    Base.pkgversion(IntervalArithmetic))

println()
println("=== validated_integ3 METHODS ===")
println(methods(VI.validated_integ3))

println()
println("=== reconstruct_rho METHODS ===")
println(methods(VI.reconstruct_rho))

function rhs_test!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

println()
println("=== FRESH ODE TEST ===")

sp = TS.JetSpace(order=20, variables=[:x])

sol_test = VI.validated_integ3(
    rhs_test!,
    [interval(1.0)],
    0.0,
    1.0,
    20,
    20,
    1e-12,
    sp
)

println("Solution type: ", typeof(sol_test))
println("Solution fields: ", fieldnames(typeof(sol_test)))
println("Time data: ", sol_test.time)
println("Flowpipe length: ", length(sol_test.fp))
println("Taylor model size: ", size(sol_test.xTM))

println()
println("=== reconstruct_rho TEST ===")

try
    rho_test = VI.reconstruct_rho(sol_test)
    println("reconstruct_rho: SUCCESS")
    println("rho type: ", typeof(rho_test))
    println("rho size: ", size(rho_test))
catch err
    println("reconstruct_rho: FAILED")
    showerror(stdout, err)
    println()
end

println()
println("=== DIAGNOSTICS FINISHED ===")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


=== PACKAGE DIAGNOSTICS ===
Julia version: 1.11.9
TaylorModels version: 0.13.1
IntervalArithmetic version: 1.0.12

=== validated_integ3 METHODS ===
# 9 methods for generic function "validated_integ3" from TaylorModels.ValidatedInteg:
 [1] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, Interval{U}, 1, N}, t0::T, tmax::T, orderQ::Int64, orderT::Int64, abstol::T, localsp::JetSpace, params; maxsteps, parse_eqs, adaptive, minabstol, absorb, check_property, midrad, tail_exact, joint) where {N, T<:Real, U<:Union{AbstractFloat, Rational}}
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:171
 [2] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, Interval{U}, 1, N}, t0::T, tmax::T, orderQ::Int64, orderT::Int64, abstol::T, localsp::JetSpace; ...) where {N, T<:Real, U<:Union{AbstractFloat, Rational}}
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:171
 [3] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, 

In [ ]:



from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic
using StaticArrays

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_compare!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

function inspect_solution(label, sol)
    println("\\n--- ", label, " ---")
    println("Type: ", typeof(sol))
    println("Expansion times: ", sol.time)
    println("Number of flowpipe entries: ", length(sol.fp))

    println("Validated time domains:")
    println(VI.mince_in_time(sol; var=0, timediv=1))

    println("Flowpipe:")
    for k in eachindex(sol.fp)
        println("fp[", k, "] = ", sol.fp[k])
    end

    println("Guaranteed remainders:")
    for k in axes(sol.xTM, 2)
        println("step ", k, ": ",
            isguaranteed(sol.xTM[1,k].rem))
    end

    println("reconstruct_rho:")
    try
        rho = VI.reconstruct_rho(sol)
        println("SUCCESS; size = ", size(rho))
    catch err
        println("FAILED: ", typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("TEST 1: ordinary Vector")
sol_vector = VI.validated_integ3(
    rhs_compare!,
    [interval(1.0)],
    0.0, 1.0,
    20, 20, 1e-12,
    sp
)
inspect_solution("VECTOR", sol_vector)

println("\\nTEST 2: StaticArrays SVector")
sol_static = VI.validated_integ3(
    rhs_compare!,
    SVector(interval(1.0)),
    0.0, 1.0,
    20, 20, 1e-12,
    sp
)
inspect_solution("SVECTOR", sol_static)

println("\\nBOTH TESTS FINISHED")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


JuliaError: ArgumentError: Package StaticArrays not found in current path.
- Run `import Pkg; Pkg.add("StaticArrays")` to install the StaticArrays package.
Stacktrace:
  [1] macro expansion
    @ ./loading.jl:2296 [inlined]
  [2] macro expansion
    @ ./lock.jl:273 [inlined]
  [3] __require(into::Module, mod::Symbol)
    @ Base ./loading.jl:2271
  [4] #invoke_in_world#3
    @ ./essentials.jl:1089 [inlined]
  [5] invoke_in_world
    @ ./essentials.jl:1086 [inlined]
  [6] require(into::Module, mod::Symbol)
    @ Base ./loading.jl:2260
  [7] eval
    @ ./boot.jl:430 [inlined]
  [8] eval
    @ ./Base.jl:130 [inlined]
  [9] pyjlmodule_seval(self::Module, expr::Py)
    @ PythonCall.JlWrap ~/.julia/packages/PythonCall/trYos/src/JlWrap/module.jl:13
 [10] _pyjl_callmethod(f::Any, self_::Ptr{PythonCall.C.PyObject}, args_::Ptr{PythonCall.C.PyObject}, nargs::Int64)
    @ PythonCall.JlWrap ~/.julia/packages/PythonCall/trYos/src/JlWrap/base.jl:65
 [11] _pyjl_callmethod(o::Ptr{PythonCall.C.PyObject},

In [ ]:



from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

println("StaticArraysCore available: ",
    Base.find_package("StaticArraysCore") !== nothing)

println("StaticArrays available: ",
    Base.find_package("StaticArrays") !== nothing)

println("\\nMethods accepting StaticArraysCore:")
using TaylorModels
println(methods(TaylorModels.ValidatedInteg.validated_integ3))
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


StaticArraysCore available: false
StaticArrays available: false

Methods accepting StaticArraysCore:
# 9 methods for generic function "validated_integ3" from TaylorModels.ValidatedInteg:
 [1] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, Interval{U}, 1, N}, t0::T, tmax::T, orderQ::Int64, orderT::Int64, abstol::T, localsp::JetSpace, params; maxsteps, parse_eqs, adaptive, minabstol, absorb, check_property, midrad, tail_exact, joint) where {N, T<:Real, U<:Union{AbstractFloat, Rational}}
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:171
 [2] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, Interval{U}, 1, N}, t0::T, tmax::T, orderQ::Int64, orderT::Int64, abstol::T, localsp::JetSpace; ...) where {N, T<:Real, U<:Union{AbstractFloat, Rational}}
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:171
 [3] validated_integ3(f!, X0::StaticArraysCore.SArray{Tuple{N}, Interval{U}, 1, N}, t0::T, tmax::T, orderQ::Int

In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

println("=== INSTALL STATICARRAYS ===")
Pkg.add("StaticArrays")

using TaylorModels
using IntervalArithmetic
using StaticArrays

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))
println("IntervalArithmetic: ", Base.pkgversion(IntervalArithmetic))
println("StaticArrays: ", Base.pkgversion(StaticArrays))

function rhs_svector_test!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])
x0 = SVector(interval(1.0))

println("=== VALIDATED ODE TEST ===")

sol_static = VI.validated_integ3(
    rhs_svector_test!,
    x0,
    0.0,
    1.0,
    20,
    20,
    1e-12,
    sp
)

println("Solution type: ", typeof(sol_static))
println("Expansion times: ", sol_static.time)
println("Flowpipe length: ", length(sol_static.fp))
println("Taylor model size: ", size(sol_static.xTM))

println("=== RECONSTRUCT RHO ===")
try
    rho_static = VI.reconstruct_rho(sol_static)
    println("RECONSTRUCT_RHO: SUCCESS")
    println("rho size: ", size(rho_static))
catch err
    println("RECONSTRUCT_RHO: FAILED")
    println("Error type: ", typeof(err))
    showerror(stdout, err)
    println()
end

println("=== TEST COMPLETE ===")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`
   Resolving package versions...


=== INSTALL STATICARRAYS ===


    Updating `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA/Project.toml`
  [90137ffa] + StaticArrays v1.9.24
  No Changes to `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA/Manifest.toml`


Julia: 1.11.9
TaylorModels: 0.13.1
IntervalArithmetic: 1.0.12
StaticArrays: 1.9.24
=== VALIDATED ODE TEST ===
Solution type: TaylorModels.ValidatedInteg.TMSol3{1, Float64, Float64}
Expansion times: [0.0, 0.0, 0.5]
Flowpipe length: 3
Taylor model size: (1, 3)
=== RECONSTRUCT RHO ===
RECONSTRUCT_RHO: FAILED
Error type: AssertionError
AssertionError: reconstruct_rho: L_{k+1} ≠ τ^0 coefficient of sol[3]
(non-deterministic, or different evaluation/preconditioning calls)
=== TEST COMPLETE ===


In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_exp_test!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

sol_check = VI.validated_integ3(
    rhs_exp_test!,
    [interval(1.0)],
    0.0,
    1.0,
    20,
    20,
    1e-12,
    sp
)

println("=== SOLUTION API ===")
println("Solution length: ", length(sol_check))
println("Available properties: ", propertynames(sol_check))
println("Expansion times: ", sol_check.time)
println("Time domains: ", VI.mince_in_time(sol_check; var=0, timediv=1))

println()
println("=== AVAILABLE EVALUATION METHODS ===")
println(methods(VI.mince_in_time))
println(methods(VI.domain))

println()
println("=== PACKAGE EVALUATION FUNCTIONS ===")
for name in names(TaylorModels; all=true)
    s = String(name)
    if occursin("eval", lowercase(s)) || occursin("bound", lowercase(s))
        println(name)
    end
end

println()
println("=== TAYLOR MODEL FUNCTIONS ===")
for name in names(TaylorModels; all=true)
    s = String(name)
    if occursin("evaluate", lowercase(s)) || occursin("bound", lowercase(s))
        println(name)
    end
end

println()
println("=== EXACT REFERENCE ===")
println("exp(0.5) = ", exp(0.5))
println("exp(1.0) = ", exp(1.0))

println()
println("=== DIAGNOSTIC COMPLETE ===")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


=== SOLUTION API ===
Solution length: 3
Available properties: (:time, :fp, :xTM)
Expansion times: [0.0, 0.0, 0.5]
Time domains: Interval{Float64}[[0.0, 0.0]_com_NG, [0.0, 0.5]_com_NG, [0.5, 1.0]_com_NG]

=== AVAILABLE EVALUATION METHODS ===
# 2 methods for generic function "mince_in_time" from TaylorModels.ValidatedInteg:
 [1] mince_in_time(sol::TaylorModels.ValidatedInteg.TMSol3; var, timediv)
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/recipe.jl:42
 [2] mince_in_time(sol::TaylorModels.ValidatedInteg.TMSol; var, timediv)
     @ ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/recipe.jl:42
# 7 methods for generic function "domain" from TaylorModels:
 [1] domain(a::TaylorModels.ValidatedInteg.TMSol3, n::Int64)
     @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/TMSol.jl:69
 [2] domain(a::TaylorModels.ValidatedInteg.TMSol3)
     @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/TMSol.jl:68
 [3] domai

In [ ]:


from juliacall import Main as jl

jl.seval("""
using Pkg

PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_exp_control!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

function run_case(label; adaptive=true, joint=true, absorb=true)
    println("\\n================================")
    println("CASE: ", label)
    println("adaptive = ", adaptive)
    println("joint = ", joint)
    println("absorb = ", absorb)

    try
        sol = VI.validated_integ3(
            rhs_exp_control!,
            [interval(1.0)],
            0.0,
            1.0,
            20,
            20,
            1e-12,
            sp;
            adaptive=adaptive,
            joint=joint,
            absorb=absorb
        )

        println("Times: ", sol.time)
        println("Domains: ", VI.mince_in_time(sol; var=0, timediv=1))

        for k in axes(sol.xTM, 2)
            tm = sol.xTM[1,k]

            println("\\nStep ", k)
            println("Domain: ", tm.dom)
            println("Remainder: ", tm.rem)
            println("Remainder guaranteed: ", isguaranteed(tm.rem))
            println("Center guaranteed: ", isguaranteed(tm.x0))

            if k == 2
                V = TaylorModels.evaluate(tm, interval(0.5))
                B = TaylorModels.monotonicity_bounder(V)

                println("Evaluation at local time 0.5: ", V)
                println("Bound: ", B)
                println("Bound guaranteed: ", isguaranteed(B))
            end
        end

        println("\\nreconstruct_rho:")
        try
            rho = VI.reconstruct_rho(sol)
            println("SUCCESS: ", size(rho))
        catch err
            println("FAILED: ", typeof(err))
            showerror(stdout, err)
            println()
        end

    catch err
        println("INTEGRATION FAILED: ", typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("Julia: ", VERSION)
println("TaylorModels: ", Base.pkgversion(TaylorModels))
println("IntervalArithmetic: ", Base.pkgversion(IntervalArithmetic))

run_case("DEFAULT")
run_case("NONADAPTIVE"; adaptive=false)
run_case("NONJOINT"; joint=false)
run_case("NO_ABSORB"; absorb=false)
run_case("ALL_CONTROLS"; adaptive=false, joint=false, absorb=false)

println("\\nALL CONTROLLED TESTS FINISHED")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


Julia: 1.11.9
TaylorModels: 0.13.1
IntervalArithmetic: 1.0.12

CASE: DEFAULT
adaptive = true
joint = true
absorb = true
Times: [0.0, 0.0, 0.5]
Domains: Interval{Float64}[[0.0, 0.0]_com_NG, [0.0, 0.5]_com_NG, [0.5, 1.0]_com_NG]

Step 1
Domain: [0.0, 0.0]_com
Remainder: [0.0, 0.0]_com
Remainder guaranteed: true
Center guaranteed: true

Step 2
Domain: [0.0, 0.5]_com_NG
Remainder: [-1.16286e-19, 7.00481e-18]_com_NG
Remainder guaranteed: false
Center guaranteed: false
Evaluation at local time 0.5:  1.6487212707001282 + [-2.22161e-16, 7.00481e-18]_com_NG
Bound: [1.64872, 1.64873]_trv_NG
Bound guaranteed: false

Step 3
Domain: [0.0, 0.5]_com_NG
Remainder: [-6.3957e-20, 3.85265e-18]_com_NG
Remainder guaranteed: false
Center guaranteed: false

reconstruct_rho:
FAILED: AssertionError
AssertionError: reconstruct_rho: L_{k+1} ≠ τ^0 coefficient of sol[3]
(non-deterministic, or different evaluation/preconditioning calls)

CASE: NONADAPTIVE
adaptive = false
joint = true
absorb = true


┌ Warning: It cannot prove existence and unicity of the solution: 
│   t[0] = 0.0
│   δt = 1.0
│   rem1 = 1-element Vector{Interval{Float64}}: …
│   _success = false
└ @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:473
┌ Warning: Exiting due to unsuccessful step
│   _success = false
│   t0 = 0.0
└ @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:275


Times: [0.0]
Domains: Interval{Float64}[[0.0, 0.0]_com_NG]

Step 1
Domain: [0.0, 0.0]_com
Remainder: [0.0, 0.0]_com
Remainder guaranteed: true
Center guaranteed: true

reconstruct_rho:
FAILED: BoundsError


┌ Warning: It cannot prove existence and unicity of the solution: 
│   t[0] = 0.0
│   δt = 1.0
│   rem1 = 1-element Vector{Interval{Float64}}: …
│   _success = false
└ @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:473
┌ Warning: Exiting due to unsuccessful step
│   _success = false
│   t0 = 0.0
└ @ TaylorModels.ValidatedInteg ~/.julia/packages/TaylorModels/uhtsT/src/valid_integ/validated_integ3.jl:275


BoundsError: attempt to access 1×1 Matrix{TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}} at index [1, 2]

CASE: NONJOINT
adaptive = true
joint = false
absorb = true
Times: [0.0, 0.0, 0.5]
Domains: Interval{Float64}[[0.0, 0.0]_com_NG, [0.0, 0.5]_com_NG, [0.5, 1.0]_com_NG]

Step 1
Domain: [0.0, 0.0]_com
Remainder: [0.0, 0.0]_com
Remainder guaranteed: true
Center guaranteed: true

Step 2
Domain: [0.0, 0.5]_com_NG
Remainder: [-1.16286e-19, 7.00481e-18]_com_NG
Remainder guaranteed: false
Center guaranteed: false
Evaluation at local time 0.5:  1.6487212707001282 + [-2.22161e-16, 7.00481e-18]_com_NG
Bound: [1.64872, 1.64873]_trv_NG
Bound guaranteed: false

Step 3
Domain: [0.0, 0.5]_com_NG
Remainder: [-6.3957e-20, 3.85265e-18]_com_NG
Remainder guaranteed: false
Center guaranteed: false

reconstruct_rho:
FAILED: AssertionError
AssertionError: reconstruct_rho: L_{k+1} ≠ τ^0 coefficient of sol[3]
(non-deterministic, or different evaluation/preconditioning calls)

CASE: NO_ABSORB
adapti

In [ ]:



from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_exp_audit!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

sol = VI.validated_integ3(
    rhs_exp_audit!,
    [interval(1.0)],
    0.0, 1.0,
    20, 20, 1e-12,
    sp
)

println("=== SOLUTION INDEXING ===")
println("length(sol) = ", length(sol))
println("time = ", sol.time)
println("propertynames = ", propertynames(sol))

for k in 1:length(sol)
    println("\\n--- sol[", k, "] ---")
    try
        println(sol[k])
    catch err
        println(typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("\\n=== TIME DOMAINS ===")
for k in 1:length(sol)
    println("domain(", k, ") = ", VI.domain(sol, k))
end

println("\\n=== FLOWPIPE ===")
for k in eachindex(sol.fp)
    println("fp[", k, "] = ", sol.fp[k])
end

println("\\n=== TAYLOR MODEL CONSTANT COEFFICIENTS ===")
for k in axes(sol.xTM, 2)
    tm = sol.xTM[1,k]
    println("\\nStep ", k)
    println("time = ", sol.time[k])
    println("domain = ", tm.dom)
    println("polynomial = ", tm.pol)
    println("remainder = ", tm.rem)
    println("total remainder = ", TaylorModels.total_remainder(tm))
end

println("\\n=== ENDPOINT REFERENCE ===")
println("exp(0.0) = ", exp(0.0))
println("exp(0.5) = ", exp(0.5))
println("exp(1.0) = ", exp(1.0))

println("\\n=== AUDIT COMPLETE ===")
""")

  Activating project at `/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA`


=== SOLUTION INDEXING ===
length(sol) = 3
time = [0.0, 0.0, 0.5]
propertynames = (:time, :fp, :xTM)

--- sol[1] ---
TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}[  1.0 + [0.0, 0.0]_com + [0.0, 0.0]_com]

--- sol[2] ---
TaylorModel1{TaylorModelN{1, Float64, Float64}, Float64}[  1.0 + [0.0, 0.0]_com + ( 1.0 + [0.0, 0.0]_com) t + ( 0.5 + [0.0, 0.0]_com) t² + ( 0.16666666666666666 + [0.0, 0.0]_com) t³ + ( 0.041666666666666664 + [0.0, 0.0]_com) t⁴ + ( 0.008333333333333333 + [0.0, 0.0]_com) t⁵ + ( 0.001388888888888889 + [0.0, 0.0]_com) t⁶ + ( 0.0001984126984126984 + [0.0, 0.0]_com) t⁷ + ( 2.48015873015873e-5 + [0.0, 0.0]_com) t⁸ + ( 2.7557319223985893e-6 + [0.0, 0.0]_com) t⁹ + ( 2.7557319223985894e-7 + [0.0, 0.0]_com) t¹⁰ + ( 2.505210838544172e-8 + [0.0, 0.0]_com) t¹¹ + ( 2.08767569878681e-9 + [0.0, 0.0]_com) t¹² + ( 1.6059043836821616e-10 + [0.0, 0.0]_com) t¹³ + ( 1.1470745597729726e-11 + [0.0, 0.0]_com) t¹⁴ + ( 7.647163731819817e-13 + [0.0, 0.0]_com) t¹⁵ + ( 4.779477332387386e-1

In [ ]:

Python

from juliacall import Main as jl

jl.seval("""
using Pkg
PROJECT = "/content/drive/MyDrive/OMEGA_GR_TRANSFER_06C_JULIA"
Pkg.activate(PROJECT)

using TaylorModels
using IntervalArithmetic

VI = TaylorModels.ValidatedInteg
TS = TaylorModels.TaylorSeries

function rhs_exp_audit!(dx, x, params, t)
    dx[1] = x[1]
    return nothing
end

sp = TS.JetSpace(order=20, variables=[:x])

sol = VI.validated_integ3(
    rhs_exp_audit!,
    [interval(1.0)],
    0.0, 1.0,
    20, 20, 1e-12,
    sp
)

println("=== SOLUTION INDEXING ===")
println("length(sol) = ", length(sol))
println("time = ", sol.time)
println("propertynames = ", propertynames(sol))

for k in 1:length(sol)
    println("\\n--- sol[", k, "] ---")
    try
        println(sol[k])
    catch err
        println(typeof(err))
        showerror(stdout, err)
        println()
    end
end

println("\\n=== TIME DOMAINS ===")
for k in 1:length(sol)
    println("domain(", k, ") = ", VI.domain(sol, k))
end

println("\\n=== FLOWPIPE ===")
for k in eachindex(sol.fp)
    println("fp[", k, "] = ", sol.fp[k])
end

println("\\n=== TAYLOR MODEL CONSTANT COEFFICIENTS ===")
for k in axes(sol.xTM, 2)
    tm = sol.xTM[1,k]
    println("\\nStep ", k)
    println("time = ", sol.time[k])
    println("domain = ", tm.dom)
    println("polynomial = ", tm.pol)
    println("remainder = ", tm.rem)
    println("total remainder = ", TaylorModels.total_remainder(tm))
end

println("\\n=== ENDPOINT REFERENCE ===")
println("exp(0.0) = ", exp(0.0))
println("exp(0.5) = ", exp(0.5))
println("exp(1.0) = ", exp(1.0))

println("\\n=== AUDIT COMPLETE ===")
""")

In [ ]:


%%writefile /content/gr_boundary_test.cpp




#include <iostream>
#include <stdexcept>
#include "capd/capdlib.h"

using namespace capd;

int main() {
    try {
        // Exact rational boundary: 22/70
        DInterval boundary =
            DInterval(22) / DInterval(70);

        // Exact rational integration step: 1/10
        DInterval eta =
            DInterval(1) / DInterval(10);

        // Exact rational bump width: 13/20
        DInterval w =
            DInterval(13) / DInterval(20);

        // Outward enclosure of the boundary interval width
        DInterval boundaryWidth =
            DInterval(boundary.rightBound())
            - DInterval(boundary.leftBound());

        // Conservative maximum distance from exact boundary
        DInterval hEff = eta + boundaryWidth;

        DInterval rho = hEff / w;

        if (rho.leftBound() <= 0.0 ||
            rho.rightBound() >= 1.0) {
            throw std::runtime_error(
                "Invalid rho enclosure"
            );
        }

        DInterval rmax =
            DInterval(2) * rho - rho * rho;

        if (rmax.leftBound() <= 0.0) {
            throw std::runtime_error(
                "Invalid rmax enclosure"
            );
        }

        // Rigorous upper bound for the bump
        DInterval U =
            exp(-DInterval(1) / rmax);

        // Differential inclusion:
        // x' = e*x, e in [0,U]
        IMap f(
            "var:x;time:t;fun:0;"
        );

        IMap g(
            "par:e;var:x;time:t;fun:e*x;"
        );

        DInterval E(0.0, U.rightBound());
        g.setParameter("e", E);

        IMultiMap rhs(f, g);

        CWDiffInclSolver solver(
            rhs,
            10,
            IMaxNorm()
        );

        solver.setStep(eta);

        IVector x0(1);
        x0[0] = 1.0;

        InclRect2Set set(x0);
        set.setCurrentTime(boundary);

        // Validated propagation
        set.move(solver);

        IVector result = IVector(set);

        // Independent analytical enclosure
        DInterval expectedUpper =
            exp(DInterval(U.rightBound()) * eta);

        DInterval expected(
            1.0,
            expectedUpper.rightBound()
        );

        std::cout.precision(17);

        std::cout << "BOUNDARY = "
                  << boundary << "\n";

        std::cout << "RHO = "
                  << rho << "\n";

        std::cout << "RMAX = "
                  << rmax << "\n";

        std::cout << "BUMP UPPER = "
                  << U << "\n";

        std::cout << "CAPD RESULT = "
                  << result << "\n";

        std::cout << "ANALYTIC ENCLOSURE = "
                  << expected << "\n";

        if (!capd::vectalg::subset(expected, result)) {
            std::cerr << "VALIDATION FAILED\n";
            return 2;
        }

        std::cout << "BOUNDARY TEST PASS\n";
        return 0;
    }
    catch (const std::exception& e) {
        std::cerr << "ERROR: " << e.what() << "\n";
        return 1;
    }
}

Overwriting /content/gr_boundary_test.cpp


In [ ]:


!g++ $(/content/CAPD/build/bin/capd-config --cflags) /content/gr_boundary_test.cpp -o /content/gr_boundary_test $(/content/CAPD/build/bin/capd-config --libs)

/bin/bash: line 1: /content/CAPD/build/bin/capd-config: No such file or directory
/bin/bash: line 1: /content/CAPD/build/bin/capd-config: No such file or directory
/content/gr_boundary_test.cpp:7:10: fatal error: capd/capdlib.h: No such file or directory
    7 | #include "capd/capdlib.h"
      |          ^~~~~~~~~~~~~~~~
compilation terminated.


In [ ]:


echo "=== CAPD DIRECTORY CHECK ==="

ls -ld /content/CAPD 2>&1

echo
echo "=== CAPD CONFIG SEARCH ==="

find /content /usr/local /usr -name capd-config -type f 2>/dev/null | head -20

echo
echo "=== CAPD HEADER SEARCH ==="

find /content /usr/local /usr -path "*/capd/capdlib.h" -type f 2>/dev/null | head -20

SyntaxError: invalid syntax (1756167555.py, line 1)

In [ ]:


!echo "=== CAPD DIRECTORY CHECK ==="
!ls -ld /content/CAPD 2>&1

!echo "=== CAPD CONFIG SEARCH ==="
!find /content /usr/local /usr -name capd-config -type f 2>/dev/null | head -20

!echo "=== CAPD HEADER SEARCH ==="
!find /content /usr/local /usr -path "*/capd/capdlib.h" -type f 2>/dev/null | head -20

=== CAPD DIRECTORY CHECK ===
ls: cannot access '/content/CAPD': No such file or directory
=== CAPD CONFIG SEARCH ===
=== CAPD HEADER SEARCH ===
